# CardioFusion-XAI — calibration fitting and validation

This notebook is the executable calibration workflow. It uses held-out **validation** predictions only. It must never use test predictions to fit calibrators or prediction intervals.

- Binary classification endpoints: fit source/target-specific Platt calibrators when there are enough validation examples from both classes.
- Multiclass outputs: remain explicitly uncalibrated unless a separately validated multiclass calibration method is implemented.
- Echo EF: calculate a split-conformal absolute-residual interval from validation predictions only.
- Different dataset endpoints are never pooled into one calibrator.

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
from sklearn.metrics import brier_score_loss, log_loss

def find_root(start=None):
    current=Path(start or Path.cwd()).resolve()
    for p in [current,*current.parents]:
        if (p/"src").is_dir() and (p/"config").is_dir(): return p
        if (p/"ml-service"/"src").is_dir(): return p/"ml-service"
    raise FileNotFoundError("Could not locate ml-service.")
ML_ROOT=find_root()
if str(ML_ROOT) not in sys.path: sys.path.insert(0,str(ML_ROOT))
from src.calibration.calibrator import PlattCalibrator
from src.calibration.metrics import calibration_metrics

ALPHA=0.10
CALIBRATION_ROOT=ML_ROOT/"checkpoints"/"calibration"
CALIBRATION_ROOT.mkdir(parents=True,exist_ok=True)

## 1. Fit biomarker and ECG calibrators from validation predictions

In [2]:
def fit_binary_groups(frame, group_columns, probability_col="prediction", truth_col="true", output_subdir=None):
    required=set(group_columns)|{probability_col,truth_col}
    missing=required-set(frame.columns)
    if missing: raise ValueError(f"Prediction file missing columns: {sorted(missing)}")
    results=[]
    for key,group in frame.groupby(group_columns):
        if not isinstance(key,tuple): key=(key,)
        p=pd.to_numeric(group[probability_col],errors="coerce").to_numpy(dtype=float)
        y=pd.to_numeric(group[truth_col],errors="coerce").to_numpy(dtype=float)
        valid=np.isfinite(p)&np.isfinite(y); p=p[valid]; y=y[valid]
        name="/".join(map(str,key))
        if len(p)<10 or set(np.unique(y))!={0.0,1.0}:
            results.append({"group":name,"status":"not_fitted","n":len(p),"reason":"need >=10 finite validation records and both binary classes"})
            continue
        calibrator=PlattCalibrator().fit(p,y)
        calibrated=calibrator.predict_proba(p)
        metric=calibration_metrics(y,p,calibrated)
        if output_subdir == "biomarkers":
            if len(key) != 2:
                raise ValueError("Biomarker calibrator groups must be (dataset, target).")
            source_name, target_name = map(str, key)
            destination = CALIBRATION_ROOT / "biomarkers" / source_name
            artifact_path = destination / f"{target_name}.pkl"
        elif output_subdir == "ecg":
            if len(key) != 1:
                raise ValueError("ECG calibrator groups must contain exactly one target name.")
            target_name = str(key[0])
            destination = CALIBRATION_ROOT / "ecg"
            artifact_path = destination / f"{target_name}.pkl"
        else:
            raise ValueError(f"Unsupported calibration output directory: {output_subdir}")
        destination.mkdir(parents=True, exist_ok=True)
        calibrator.save(artifact_path)
        results.append({"group":name,"status":"fitted","artifact":str(artifact_path),**metric})
    return results

biomarker_path=ML_ROOT/"artifacts"/"metrics"/"biomarkers"/"ft_transformer_validation_predictions.csv"
if biomarker_path.is_file():
    frame=pd.read_csv(biomarker_path)
    frame=frame[frame["split"].astype(str).str.lower().isin(["validation","val"])]
    frame=frame[frame["output_type"]=="binary_probability"]
    biomarker_results=fit_binary_groups(frame,["dataset","target"],output_subdir="biomarkers")
    display(pd.DataFrame(biomarker_results))
else:
    print("No FT-Transformer biomarker validation predictions found:",biomarker_path)

ecg_path=ML_ROOT/"artifacts"/"metrics"/"ecg"/"ecg_transformer"/"ptbxl_val_predictions.csv"
if ecg_path.is_file():
    ecg=pd.read_csv(ecg_path)
    rows=[]
    for col in [c for c in ecg.columns if c.endswith("_prob") and c[:-5]+"_true" in ecg.columns]:
        target=col[:-5]
        part=ecg[[col,target+"_true"]].rename(columns={col:"prediction",target+"_true":"true"})
        part["target"]=target
        rows.append(part)
    if rows:
        ecg_long=pd.concat(rows,ignore_index=True)
        ecg_results=fit_binary_groups(ecg_long,["target"],output_subdir="ecg")
        display(pd.DataFrame(ecg_results))
else:
    print("No ECG Transformer validation prediction file found:",ecg_path)

,group,status,artifact,raw_brier,calibrated_brier,raw_log_loss,calibrated_log_loss,raw_ece,calibrated_ece
0,framingham/ten_year_chd_risk,fitted,D:\Programming\VS Code\Projects\CardioFusion-X...,0.218139,0.118936,0.626238,0.387519,0.300826,0.022459
1,mi_complications/a_v_blok,fitted,D:\Programming\VS Code\Projects\CardioFusion-X...,0.139443,0.019114,0.449551,0.092685,0.317028,0.000004
2,mi_complications/dressler,fitted,D:\Programming\VS Code\Projects\CardioFusion-X...,0.227370,0.034041,0.647550,0.152571,0.420881,0.000051
3,mi_complications/fibr_jelud,fitted,D:\Programming\VS Code\Projects\CardioFusion-X...,0.233282,0.045344,0.664763,0.176675,0.406826,0.006880
4,mi_complications/fibr_preds,fitted,D:\Programming\VS Code\Projects\CardioFusion-X...,0.246616,0.088440,0.686677,0.312193,0.395208,0.016373
5,mi_complications/jelud_tah,fitted,D:\Programming\VS Code\Projects\CardioFusion-X...,0.151616,0.039361,0.458349,0.162232,0.259722,0.008691
6,mi_complications/o_l_post,fitted,D:\Programming\VS Code\Projects\CardioFusion-X...,0.156856,0.059258,0.495450,0.221679,0.293142,0.022126
7,mi_complications/p_im_sten,fitted,D:\Programming\VS Code\Projects\CardioFusion-X...,0.255000,0.090096,0.703136,0.320166,0.399219,0.015827
8,mi_complications/preds_tah,fitted,D:\Programming\VS Code\Projects\CardioFusion-X...,0.175997,0.007777,0.541982,0.045515,0.406396,0.000054
9,mi_complications/razriv,fitted,D:\Programming\VS Code\Projects\CardioFusion-X...,0.135754,0.032329,0.432359,0.136734,0.271488,0.005633


,group,status,artifact,raw_brier,calibrated_brier,raw_log_loss,calibrated_log_loss,raw_ece,calibrated_ece
0,CD,fitted,D:\Programming\VS Code\Projects\CardioFusion-X...,0.079982,0.079859,0.269106,0.268669,0.010129,0.015698
1,HYP,fitted,D:\Programming\VS Code\Projects\CardioFusion-X...,0.068484,0.067490,0.232929,0.229387,0.018824,0.011211
2,MI,fitted,D:\Programming\VS Code\Projects\CardioFusion-X...,0.104303,0.096372,0.336095,0.311369,0.065044,0.009826
3,NORM,fitted,D:\Programming\VS Code\Projects\CardioFusion-X...,0.098293,0.090627,0.317801,0.291860,0.062362,0.023835
4,RHYTHM_AFIB,fitted,D:\Programming\VS Code\Projects\CardioFusion-X...,0.026201,0.025572,0.090968,0.087966,0.012763,0.009128
5,RHYTHM_AFLT,fitted,D:\Programming\VS Code\Projects\CardioFusion-X...,0.002781,0.002620,0.012487,0.010550,0.004465,0.002987
6,RHYTHM_BIGU,fitted,D:\Programming\VS Code\Projects\CardioFusion-X...,0.003646,0.003622,0.023333,0.022833,0.002014,0.000002
7,RHYTHM_PACE,fitted,D:\Programming\VS Code\Projects\CardioFusion-X...,0.006675,0.006590,0.031074,0.030617,0.004118,0.002326
8,RHYTHM_PSVT,fitted,D:\Programming\VS Code\Projects\CardioFusion-X...,0.001340,0.001343,0.007618,0.006916,0.000759,0.000006
9,RHYTHM_SARRH,fitted,D:\Programming\VS Code\Projects\CardioFusion-X...,0.033647,0.033652,0.147700,0.147505,0.002986,0.002530


## 2. Fit EF split-conformal absolute-error interval

The quantile is estimated using validation residuals. This produces a marginal prediction interval under exchangeability assumptions; it is not a patient-specific epistemic uncertainty estimate and should be assessed for coverage on a separate test cohort.

In [3]:
echo_candidates=[
    ML_ROOT/"artifacts"/"metrics"/"echo"/"panecho"/"finetuned"/"validation_predictions.csv",
    ML_ROOT/"artifacts"/"metrics"/"echo"/"panecho"/"finetuned"/"predictions_validation.csv",
]
echo_path=next((p for p in echo_candidates if p.is_file()),None)
if echo_path is None:
    print("Echo validation prediction file not found. Run the existing PanEcho fine-tuning/evaluation notebooks first.")
else:
    echo=pd.read_csv(echo_path)
    pred_col=next((c for c in ["EF_pred","ef_pred","prediction"] if c in echo.columns),None)
    true_col=next((c for c in ["EF_true","ef_true","EF","ef"] if c in echo.columns),None)
    if pred_col is None or true_col is None: raise ValueError(f"Cannot identify EF prediction/truth columns in {echo_path}; columns={list(echo.columns)}")
    pred=pd.to_numeric(echo[pred_col],errors="coerce").to_numpy(dtype=float)
    true=pd.to_numeric(echo[true_col],errors="coerce").to_numpy(dtype=float)
    valid=np.isfinite(pred)&np.isfinite(true)
    residuals=np.abs(true[valid]-pred[valid])
    if len(residuals)<20: raise ValueError("At least 20 finite validation EF residuals are required for this interval estimate.")
    # Finite-sample conformal quantile index.
    rank=min(len(residuals), int(np.ceil((len(residuals)+1)*(1-ALPHA))))
    q=float(np.sort(residuals)[rank-1])
    payload={"absolute_error_quantile":q,"nominal_coverage":1-ALPHA,"alpha":ALPHA,"n_validation":int(len(residuals)),
             "source_file":str(echo_path),"method":"split_conformal_absolute_residual_quantile"}
    out=CALIBRATION_ROOT/"echo"/"ef_absolute_error_quantile.json"
    out.parent.mkdir(parents=True,exist_ok=True); out.write_text(json.dumps(payload,indent=2),encoding="utf-8")
    print(json.dumps(payload,indent=2))

Echo validation prediction file not found. Run the existing PanEcho fine-tuning/evaluation notebooks first.


## 3. Calibration audit and safeguards

Review that all input CSVs contain validation predictions only. Calibration is performed per endpoint and per source, not across dataset populations. A successfully fitted calibrator does not make a source endpoint clinically generalizable; it only adjusts probabilities for the source-defined target under the calibration population assumptions.

In [4]:
calibration_inventory=[]
for path in CALIBRATION_ROOT.rglob("*.pkl"):
    calibration_inventory.append({"artifact":str(path.relative_to(CALIBRATION_ROOT)),"size_bytes":path.stat().st_size})
for path in CALIBRATION_ROOT.rglob("*.json"):
    calibration_inventory.append({"artifact":str(path.relative_to(CALIBRATION_ROOT)),"size_bytes":path.stat().st_size})
display(pd.DataFrame(calibration_inventory))
print("Calibration root:",CALIBRATION_ROOT)

,artifact,size_bytes
0,biomarker_acute_mi.pkl,761
1,biomarker\framingham_ten_year_chd_risk.pkl,761
2,biomarker\mi_complications_post_mi_af.pkl,761
3,biomarker\mi_complications_post_mi_angina.pkl,761
4,biomarker\mi_complications_post_mi_av_block.pkl,761
5,biomarker\mi_complications_post_mi_chronic_hf.pkl,761
6,biomarker\mi_complications_post_mi_dressler.pkl,761
7,biomarker\mi_complications_post_mi_myocardial_...,761
8,biomarker\mi_complications_post_mi_recurrent_m...,761
9,biomarker\mi_complications_post_mi_svt.pkl,761


Calibration root: D:\Programming\VS Code\Projects\CardioFusion-XAI\ml-service\checkpoints\calibration
